# Week 2 — SQL Validation of Event-Level KSI Data

Goal: validate the cleaned Toronto KSI event-level dataset using SQL queries.

Input dataset: `data/interim/ksi_events.csv`

Grain: one row = one collision event.

Primary key: `collision_id`.

The SQL checks are intended to independently verify dataset structure,
temporal coverage, missingness, uniqueness, and basic event distributions.

No machine-learning target or feature set is defined at this stage.

## Validation Plan

The cleaned event-level dataset will be checked for:

1. total row count and unique collision IDs;
2. duplicate `collision_id` values;
3. collision counts by year;
4. collision counts by month;
5. collision-class distribution;
6. missing location coordinates;
7. missing road-context information;
8. temporal coverage.

The SQL results should agree with the corresponding Pandas checks performed
during event-level cleaning.

In [2]:
import pandas as pd

sql_df = pd.read_csv("../data/interim/ksi_events.csv")

sql_df.head()

,collision_id,accdate,stname1,stname2,stname3,acclass,accloc,traffictl,impactype,visible,light,rdsfcond,road_class,longitude,latitude,division,neighbourhood
0,2006:893184,2006-01-01 02:36:00,WOODBINE AVE,O CONNOR DR,NaN,Non-Fatal Injury,Intersection-Related,No Control,Approaching Head On,Clear,Dark,Wet,Major Arterial,-79.318798,43.699594,55 Division,Woodbine-Lumsden
1,2006:909646,2006-01-01 03:15:00,DANFORTH AVE,WEST LYNN AVE,NaN,Non-Fatal Injury,Intersection-Related,No Control,SMV Other,Snow,Dark with Artificial Lighting,Slush,Major Arterial,-79.316189,43.684873,55 Division,Woodbine Corridor
2,2026:6000734335,2026-04-10 06:10:00,ALBION RD,FINCH AV W,NaN,Non-Fatal Injury,Intersection-Related,Traffic Signal,SMV Other,Rain,Dark,Wet,NaN,-79.589200,43.741099,23 Division,Mount Olive-Silverstone-Jamestown
3,2026:6001470477,2026-07-15 18:33:00,28 CARNFORTH RD,LAWRENCE AV E,5 m East of,Non-Fatal Injury,Non-Intersection,Unknown,Approaching Head On,Clear,Dusk,Dry,NaN,-79.315404,43.739802,33 Division,Victoria Village
4,2026:6001482161,2026-07-17 09:39:00,HIGHWAY 2A E,MEADOWVALE RD,80 m East of,Non-Fatal Injury,Non-Intersection,No Control,Rear End,Clear,Daylight,Dry,NaN,-79.169880,43.780539,43 Division,Centennial Scarborough


In [3]:
sql_df.shape

(7620, 17)

In [4]:
import sqlite3

conn = sqlite3.connect(":memory:")

sql_df.to_sql(
    "ksi_events",
    conn,
    if_exists="replace",
    index=False
)

7620

In [5]:
pd.read_sql_query(
    "SELECT COUNT(*) AS row_count FROM ksi_events;",
    conn
)

,row_count
0,7620


In [6]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT collision_id) AS unique_collision_ids
FROM ksi_events;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_collision_ids
0,7620,7620


In [7]:
query = """
SELECT
    collision_id,
    COUNT(*) AS row_count
FROM ksi_events
GROUP BY collision_id
HAVING COUNT(*) > 1;
"""

pd.read_sql_query(query, conn)

,collision_id,row_count


In [8]:
query = """
SELECT
    strftime('%Y', accdate) AS year,
    COUNT(*) AS collision_count
FROM ksi_events
GROUP BY year
ORDER BY year;
"""

pd.read_sql_query(query, conn)

,year,collision_count
0,2006,481
1,2007,453
2,2008,417
3,2009,438
4,2010,400
5,2011,400
6,2012,453
7,2013,431
8,2014,350
9,2015,350


In [9]:
query = """
SELECT
    COUNT(*) AS missing_coordinates
FROM ksi_events
WHERE longitude IS NULL
   OR latitude IS NULL;
"""

pd.read_sql_query(query, conn)

,missing_coordinates
0,1


## Validation Summary

The SQL validation confirms the main invariants established during Pandas
cleaning:

- the table contains 7,620 collision events;
- all 7,620 `collision_id` values are unique;
- no duplicate collision IDs are present;
- collision events cover the period from 2006 through 2026;
- one collision event is missing spatial coordinates.

The SQL checks are consistent with the Pandas-based data-quality audit.